# To read dataframe, just use ```polars.read_*()``` or ```polars.scan_*()``` functions, then convert to datar tibble if needed.

In [1]:
import datar.all as dr
import polars as pl
from polars import col as c

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
pl.Config.set_tbl_cell_alignment("CENTER")

data_dir = next(Path("/home/").glob("**/datar-polars/data"))

# <span style="color:#1E90FF">1. Read from a CSV file</span>

In [2]:
tb_csv = dr.tibble(
    pl.read_csv(data_dir/'air_quality_no2_long.csv')
)

print(tb_csv.head())

shape: (5, 7)
┌───────┬─────────┬───────────────────────────┬──────────┬───────────┬───────┬───────┐
│  city ┆ country ┆          date.utc         ┆ location ┆ parameter ┆ value ┆  unit │
│  ---  ┆   ---   ┆            ---            ┆    ---   ┆    ---    ┆  ---  ┆  ---  │
│  str  ┆   str   ┆            str            ┆    str   ┆    str    ┆  f64  ┆  str  │
╞═══════╪═════════╪═══════════════════════════╪══════════╪═══════════╪═══════╪═══════╡
│ Paris ┆    FR   ┆ 2019-06-21 00:00:00+00:00 ┆  FR04014 ┆    no2    ┆  20.0 ┆ µg/m³ │
│ Paris ┆    FR   ┆ 2019-06-20 23:00:00+00:00 ┆  FR04014 ┆    no2    ┆  21.8 ┆ µg/m³ │
│ Paris ┆    FR   ┆ 2019-06-20 22:00:00+00:00 ┆  FR04014 ┆    no2    ┆  26.5 ┆ µg/m³ │
│ Paris ┆    FR   ┆ 2019-06-20 21:00:00+00:00 ┆  FR04014 ┆    no2    ┆  24.9 ┆ µg/m³ │
│ Paris ┆    FR   ┆ 2019-06-20 20:00:00+00:00 ┆  FR04014 ┆    no2    ┆  21.4 ┆ µg/m³ │
└───────┴─────────┴───────────────────────────┴──────────┴───────────┴───────┴───────┘


# <span style="color:#1E90FF">2. Read from an EXCEL file</span>

In [3]:
tb_excel = dr.tibble(
    pl.read_excel(
        source=data_dir/"emp_sheetname.xlsx",
        sheet_name='emp'
    )
)

print(tb_excel.head())

shape: (5, 5)
┌─────┬──────────┬────────┬────────────┬────────────┐
│  id ┆   name   ┆ salary ┆ start_date ┆    dept    │
│ --- ┆    ---   ┆   ---  ┆     ---    ┆     ---    │
│ str ┆    str   ┆   f64  ┆    date    ┆     str    │
╞═════╪══════════╪════════╪════════════╪════════════╡
│  1  ┆   Rick   ┆  623.3 ┆ 2012-01-01 ┆     IT     │
│  2  ┆    Dan   ┆  515.2 ┆ 2013-09-23 ┆ Operations │
│  3  ┆ Michelle ┆  611.0 ┆ 2014-11-15 ┆     IT     │
│  4  ┆   Ryan   ┆  729.0 ┆ 2014-05-11 ┆     HR     │
│     ┆   Gary   ┆ 843.25 ┆ 2015-03-27 ┆   Finance  │
└─────┴──────────┴────────┴────────────┴────────────┘


# <span style="color:#1E90FF">3. Read with more customizations</span>

In [47]:
tb_pokemon = (
    pl.scan_csv( # scan_csv() results in a LazyDataframe, so some features are not supported
        source=data_dir/'pokemon.csv',
        schema_overrides={
            "Type 1": pl.Categorical,
            "Type 2": pl.Categorical,
            "Generation": pl.Categorical,
            "Legendary": pl.Boolean
        }
    )
    .drop("#")
    .rename(lambda col: pl.Series([col]).str.replace(r"\s+", "_",).str.replace(".", "", literal=True).item())
    .collect() # realize the lazy dataframe to use pipe with subscriptable "f"
    .pipe(lambda f: f.with_columns(Generation = c("Generation").cast(pl.Enum(f["Generation"].unique().sort().to_list()))))
)

print(
    tb_pokemon
    >> dr.slice_head(5)
)

shape: (5, 12)
┌───────────────────────┬────────┬────────┬───────┬───┬────────┬───────┬────────────┬───────────┐
│          Name         ┆ Type_1 ┆ Type_2 ┆ Total ┆ … ┆ Sp_Def ┆ Speed ┆ Generation ┆ Legendary │
│          ---          ┆   ---  ┆   ---  ┆  ---  ┆   ┆   ---  ┆  ---  ┆     ---    ┆    ---    │
│          str          ┆   cat  ┆   cat  ┆  i64  ┆   ┆   i64  ┆  i64  ┆    enum    ┆    bool   │
╞═══════════════════════╪════════╪════════╪═══════╪═══╪════════╪═══════╪════════════╪═══════════╡
│       Bulbasaur       ┆  Grass ┆ Poison ┆  318  ┆ … ┆   65   ┆   45  ┆      1     ┆   false   │
│        Ivysaur        ┆  Grass ┆ Poison ┆  405  ┆ … ┆   80   ┆   60  ┆      1     ┆   false   │
│        Venusaur       ┆  Grass ┆ Poison ┆  525  ┆ … ┆   100  ┆   80  ┆      1     ┆   false   │
│ VenusaurMega Venusaur ┆  Grass ┆ Poison ┆  625  ┆ … ┆   120  ┆   80  ┆      1     ┆   false   │
│       Charmander      ┆  Fire  ┆  null  ┆  309  ┆ … ┆   50   ┆   65  ┆      1     ┆   false   │
└────